In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

csv_path = Path("multivariate_odds_ratios_95CI_full_sample.csv")

# Raw column names -> display labels used in the manuscript figures/tables.
LABEL_MAP = {
    "Depression": "Depression symptoms",
    "SelfEsteem": "Self-esteem issues",
    "FEMOTION": "Emotional symptoms",
    "FPEER": "Peer problems",
    "child_ilness_Yes": "Childhood illness: yes",
    "sex_male": "Female sex",
    "mKessler": "Parent 1 psychological distress",
    "BMI": "BMI",
    "mDepression_Yes": "Parent 1 depression: yes",
    "Obesity_obese": "Weight status: obese",
    "child_alcohol_many": "Alcohol use: many occasions",
    "mExtravert": "Parent 1 extraversion",
    "mNeurotic": "Parent 1 neuroticism",
    "child_alcohol_some": "Alcohol use: some occasions",
    "cog_risk_taking": "Risk-taking",
    "knockedout_Yes": "History of loss of consciousness: yes",
    "mReligion_hindu": "Parent 1 religion: Hindu",
    "fExtravert": "Parent 2 extraversion",
    "child_cannabis_more than 5": "Cannabis use: more than 5 times",
    "mEthnicity_non-white": "Parent 1 ethnicity: non-White",
    "FCONDUCT": "Conduct problems",
    "mEdu_higher edu": "Parent 1 education: higher education",
    "mConscienc": "Parent 1 conscientiousness",
    "cog_risk_adjustment": "Risk adjustment",
    "mEmploy_management": "Parent 1 employment: managerial",
    "mOpenness": "Parent 1 openness",
    "Obesity_overweight": "Weight status: overweight",
    "child_cannabis_one to four": "Cannabis use: 1–4 times",
    "mReligion_muslim": "Parent 1 religion: Muslim",
    "FPROSOC": "Prosocial behaviour",
    "fOpenness": "Parent 2 openness",
    "fAlcohol_binary_high risk": "Parent 2 alcohol use: high risk",
    "fDepression_Yes": "Parent 2 depression: yes",
    "banghead_Yes": "History of head injury: yes",
    "mean_acc_24h": "Mean 24-hour acceleration",
    "mvpa_acc_5sec": "Moderate-to-vigorous physical activity",
    "fKessler": "Parent 2 psychological distress",
    "mAlcohol_binary_high risk": "Parent 1 alcohol use: high risk",
    "mEdu_overseas": "Parent 1 education: overseas qualification",
    "fAgree": "Parent 2 agreeableness",
    "m5_hour_start": "Most active 5 hour block start time",
    "mEmploy_s-emp": "Parent 1 employment: self-employed",
    "fConscienc": "Parent 2 conscientiousness",
    "cog_decision_making": "Decision-making",
    "sexualassault_Yes": "History of sexual assault: yes",
    "fNeurotic": "Parent 2 neuroticism",
    "mAgree": "Parent 1 agreeableness",
    "fAlcohol": "Parent 2 alcohol use",
    "child_specneeds_Yes": "Special educational needs: yes",
    "child_autism_Yes": "Autism diagnosis: yes",
    "mEmploy_semi-rou and routine": "Parent 1 employment: semi-routine or routine",
    "cog_word_activity": "Word activity score",
    "FHYPER": "Hyperactivity/inattention",
    "l5_hour_start": "Least active 5h block start time",
    "child_ADHD_Yes": "ADHD diagnosis: yes",
    "mReligion_none": "Parent 1 religion: none",
    "mEmploy_lo sup and tech": "Parent 1 employment: lower supervisory or technical",
    "mReligion_christian": "Parent 1 religion: Christian",
    "mAlcohol": "Parent 1 alcohol use",
    "mAge": "Parent 1 age",
    "mReligion_sikh": "Parent 1 religion: Sikh",
    "mReligion_other": "Parent 1 religion: other",
}

df = pd.read_csv(csv_path)

required_columns = {"Feature", "OR", "CI_low", "CI_high", "p_value"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}\n"
        f"Available columns: {list(df.columns)}"
    )

df["Feature_raw"] = df["Feature"]
df["Feature"] = df["Feature"].map(LABEL_MAP).fillna(df["Feature"])

numeric_columns = ["OR", "CI_low", "CI_high", "p_value"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=["Feature", "OR", "CI_low", "CI_high"]).copy()
df = df[(df["OR"] > 0) & (df["CI_low"] > 0) & (df["CI_high"] > 0)].copy()

# Model was fit with male sex as the predictor (female = reference); flip it
# so female sex is reported instead, keeping the direction consistent with
# the other risk factors. Inverting a binary predictor's reference category
# is just 1/OR (CI bounds swap and invert accordingly); p-value is unchanged.
sex_mask = df["Feature_raw"] == "sex_male"
if sex_mask.any():
    old_or = df.loc[sex_mask, "OR"].iloc[0]
    old_ci_low = df.loc[sex_mask, "CI_low"].iloc[0]
    old_ci_high = df.loc[sex_mask, "CI_high"].iloc[0]
    df.loc[sex_mask, "OR"] = 1 / old_or
    df.loc[sex_mask, "CI_low"] = 1 / old_ci_high
    df.loc[sex_mask, "CI_high"] = 1 / old_ci_low


def significance_stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    return ""


df["stars"] = df["p_value"].apply(significance_stars)
df["significant"] = df["p_value"] < 0.05

# CSV is sorted by ascending p-value; reverse it so the strongest
# association ends up at the top of the plot.
plot_df = df.iloc[::-1].reset_index(drop=True)

# "Cannabis use: more than 5 times" and "Cannabis use: 1-4 times" are two
# levels of the same categorical variable (same reference: never used), so
# keep them adjacent regardless of where their p-values would place them.
cannabis_high = "Cannabis use: more than 5 times"
cannabis_low = "Cannabis use: 1–4 times"
if cannabis_high in plot_df["Feature"].values and cannabis_low in plot_df["Feature"].values:
    low_row = plot_df[plot_df["Feature"] == cannabis_low]
    remaining = plot_df[plot_df["Feature"] != cannabis_low]
    high_pos = remaining.index[remaining["Feature"] == cannabis_high][0]
    before = remaining.loc[:high_pos - 1]
    after = remaining.loc[high_pos:]
    plot_df = pd.concat([before, low_row, after]).reset_index(drop=True)

y_positions = np.arange(len(plot_df))
lower_errors = plot_df["OR"] - plot_df["CI_low"]
upper_errors = plot_df["CI_high"] - plot_df["OR"]

figure_height = max(6, 0.48 * len(plot_df) + 1.5)
fig, ax = plt.subplots(figsize=(11, figure_height))

non_significant = ~plot_df["significant"]
ax.errorbar(
    x=plot_df.loc[non_significant, "OR"],
    y=y_positions[non_significant],
    xerr=np.vstack([lower_errors.loc[non_significant], upper_errors.loc[non_significant]]),
    fmt="o", markersize=5, markerfacecolor="white", markeredgecolor="black",
    ecolor="black", elinewidth=1, capsize=3, linestyle="none",
)

significant = plot_df["significant"]
ax.errorbar(
    x=plot_df.loc[significant, "OR"],
    y=y_positions[significant],
    xerr=np.vstack([lower_errors.loc[significant], upper_errors.loc[significant]]),
    fmt="s", markersize=6, markerfacecolor="black", markeredgecolor="black",
    ecolor="black", elinewidth=1.2, capsize=3, linestyle="none",
)

ax.axvline(x=1, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_df["Feature"], fontsize=10)

x_ticks = [0.1, 0.25, 0.5, 1, 2, 5, 10]
ax.set_xticks(x_ticks)
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{value:g}"))
ax.set_xlim(0.1, 10)
ax.set_xlabel("Odds ratio (95% confidence interval)", fontsize=11)

ax.grid(axis="x", which="major", linestyle=":", linewidth=0.7, alpha=0.5)
ax.grid(axis="y", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)

for row_number, row in plot_df.iterrows():
    or_ci_text = f"{row['OR']:.2f} ({row['CI_low']:.2f}–{row['CI_high']:.2f})"
    p_text = "<.001" if row["p_value"] < 0.001 else f"{row['p_value']:.3f}"

    ax.text(1.02, row_number, or_ci_text, transform=ax.get_yaxis_transform(),
            ha="left", va="center", fontsize=9, clip_on=False)
    ax.text(1.52, row_number, f"{p_text} {row['stars']}", transform=ax.get_yaxis_transform(),
            ha="left", va="center", fontsize=9, clip_on=False)

ax.text(1.02, 1.015, "Adjusted OR (95% CI)", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=10, fontweight="bold")
ax.text(1.52, 1.015, "p-value", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=10, fontweight="bold")

fig.text(0.995, 0.01, "* p < .05, ** p < .01, *** p < .001", ha="right", va="bottom", fontsize=8)
fig.subplots_adjust(left=0.36, right=0.72, top=0.94, bottom=0.08)

output_png = Path("multivariable_forest_plot_full_sample.png")
output_pdf = Path("multivariable_forest_plot_full_sample.pdf")
output_clean_csv = Path("multivariable_results_clean_names_full_sample.csv")

fig.savefig(output_png, dpi=300, bbox_inches="tight")
fig.savefig(output_pdf, bbox_inches="tight")
plot_df.to_csv(output_clean_csv, index=False)
plt.show()
